# Red Team Eval — Google Colab

Runs the evaluation pipeline on Colab's GPU using Ollama for local inference.

**Before running:**
1. Set runtime to GPU: *Runtime → Change runtime type → T4 GPU*
2. Set `PROJECT_PATH` to where `eval.py` and `data/` live in your Google Drive
3. Edit the **Eval configuration** cell below — it writes `config.py` automatically


In [96]:
# ── Project path ───────────────────────────────────────────────────────────────
# Only eval.py and the data/ folder need to exist here.
PROJECT_PATH = '/content/drive/MyDrive/BA_experiments'

In [97]:
# ── Mount Google Drive ─────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ── Inspect log status ─────────────────────────────────────────────────────────
# Run this at any time after mounting Drive to see which experiments are in each
# log file and how many samples completed. Useful after an interrupted run.
# Requires only: Drive mounted + PROJECT_PATH set (cell 1).

import subprocess
subprocess.run(['pip', 'install', '-q', 'zipfile-zstd'], check=True)

import json
import zipfile
import zipfile_zstd  # registers zstd (method 93) codec into zipfile
from pathlib import Path

LOGS_DIR = Path(PROJECT_PATH) / 'logs'

STATUS_ICON = {
    'success':   '✓',
    'error':     '✗',
    'cancelled': '⚠',
    'started':   '…',
}

def _read_header(path: Path) -> tuple[dict | None, str | None]:
    """Return (header_dict, None) on success, or (None, error_message) on failure."""
    try:
        with zipfile.ZipFile(path) as z:
            with z.open('header.json') as f:
                return json.load(f), None
    except Exception as e:
        return None, f'{type(e).__name__}: {e}'

def _get_grader(ev: dict) -> str:
    """Extract grader model name from task_args (old format) or scorer options (new format)."""
    # New format: grader is in scorer options
    for scorer in ev.get('scorers', []):
        gid = scorer.get('options', {}).get('grader_model_id', '')
        if gid:
            return gid.removeprefix('openai/')
    # Old format: grader was a task arg
    gid = ev.get('task_args', {}).get('grader', '')
    if gid:
        return gid.removeprefix('openai/')
    return '?'

# ── Collect info from every .eval file ────────────────────────────────────────
if not LOGS_DIR.exists():
    print(f'No logs directory found at {LOGS_DIR}')
else:
    log_files = sorted(LOGS_DIR.glob('*.eval'))
    print(f'Scanning {LOGS_DIR} ...')
    print(f'Found {len(log_files)} log file(s).\n')

    rows = []
    errors = []
    for path in log_files:
        h, err = _read_header(path)
        if h is None:
            errors.append((path.name, err))
            continue

        status = h.get('status', '?')
        ev     = h.get('eval', {})
        args   = ev.get('task_args', {})
        ds     = ev.get('dataset', {})
        cfg    = ev.get('config', {})
        res    = h.get('results') or {}

        model   = args.get('model', ev.get('model', '?')).removeprefix('openai/')
        grader  = _get_grader(ev)
        lang    = args.get('language', '?')
        epochs  = cfg.get('epochs', '?')

        ds_loc  = ds.get('location', '')
        split   = next(
            (s for s in ('1_sample', '10_sample', '50_sample', '100_sample', '300_sample', 'all') if s in ds_loc),
            ds.get('name', '?'),
        )

        total     = res.get('total_samples')
        completed = res.get('completed_samples')

        if total is None and ds.get('samples') and isinstance(epochs, int):
            total = ds['samples'] * epochs

        rows.append({
            'file':      path.name,
            'status':    status,
            'model':     model,
            'grader':    grader,
            'lang':      lang,
            'split':     split,
            'epochs':    epochs,
            'completed': completed,
            'total':     total,
        })

    # ── Print table ───────────────────────────────────────────────────────────
    if rows:
        w_model  = max(len(r['model'])  for r in rows)
        w_grader = max(len(r['grader']) for r in rows)
        w_split  = max(len(r['split'])  for r in rows)

        header = (
            f"{'#':>3}  {'St':2}  {'Model':<{w_model}}  {'Grader':<{w_grader}}  "
            f"{'Lang':<4}  {'Dataset':<{w_split}}  {'Ep':>2}  {'Done / Total':>18}"
        )
        sep = '─' * len(header)
        print(header)
        print(sep)

        status_counts = {}
        for i, r in enumerate(rows, 1):
            icon  = STATUS_ICON.get(r['status'], '?')
            total = r['total']
            done  = r['completed']

            if total and done is not None:
                pct    = done / total * 100
                done_s = f"{done:>4} / {total:<4}  ({pct:5.1f}%)"
            elif done is not None:
                done_s = f"{done:>4} / {'?':<4}"
            else:
                done_s = f"{'?':>4} / {'?':<4}"

            print(
                f"{i:>3}  {icon}   {r['model']:<{w_model}}  {r['grader']:<{w_grader}}  "
                f"{r['lang']:<4}  {r['split']:<{w_split}}  {r['epochs']:>2}  {done_s:>18}"
            )

            status_counts[r['status']] = status_counts.get(r['status'], 0) + 1

        print(sep)
        print('\nStatus summary:')
        for st, count in sorted(status_counts.items()):
            icon = STATUS_ICON.get(st, '?')
            print(f'  {icon}  {st:<12}: {count}')

    if errors:
        print(f'\nCould not read {len(errors)} file(s):')
        for name, err in errors:
            print(f'  {name}')
            print(f'    -> {err}')

In [98]:
# ── GPU check ──────────────────────────────────────────────────────────────────
# Verify Colab assigned a GPU. If this fails, go to:
#   Runtime → Change runtime type → Hardware accelerator → T4 GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null \
    && echo 'GPU OK' \
    || echo 'WARNING: No GPU detected — change runtime type before continuing'

Tesla T4, 15360 MiB
GPU OK


In [ ]:
# ── Eval configuration ─────────────────────────────────────────────────────────
# Edit these values. This cell writes src/config.py to your Drive so you never
# need to touch the file in the project folder.
# Models use the 'openai/' prefix (inspect-ai's Ollama compatibility shim).

DATASET = "100_sample"       # "all" | "1_sample" | "10_sample" | "100_sample" | "300_sample"

MODELS = [
    "openai/gemma3:270m",
    # "openai/deepseek-r1:1.5b",
    # "openai/mistral:7b"
    # "openai/MichelRosselli/apertus:8b-instruct-2509-q4_k_m",
]

GRADER_MODEL = [
    # "openai/gemma3:270m",
    # "openai/deepseek-r1:1.5b",
    "openai/mistral:7b"
    # "openai/MichelRosselli/apertus:8b-instruct-2509-q4_k_m",
]

LANGUAGES = [
    "en",
    "de",
    "ar",
    "ch",
    "ta",
]

N_RUNS = 5
CONFIG = {"seed": 42, "temperature": 0.0, "top_p": 1.0, "max_tokens": 500}
GRADER_CONFIG = {"seed": 42, "temperature": 0.0, "top_p": 1.0, "max_tokens": 300}

# ── Write config.py ────────────────────────────────────────────────────────────
import json, textwrap

_cfg = textwrap.dedent(f"""\
from dotenv import load_dotenv
load_dotenv()

DATASET       = {json.dumps(DATASET)}
MODELS        = {json.dumps(MODELS, indent=4)}
GRADER_MODEL  = {json.dumps(GRADER_MODEL, indent=4)}
LANGUAGES     = {json.dumps(LANGUAGES, indent=4)}
N_RUNS        = {json.dumps(N_RUNS, indent=4)}
CONFIG        = {json.dumps(CONFIG, indent=4)}
GRADER_CONFIG = {json.dumps(GRADER_CONFIG, indent=4)}
""")

with open(f'{PROJECT_PATH}/config.py', 'w') as f:
    f.write(_cfg)
print('config.py written')

In [100]:
# ── Install Ollama ─────────────────────────────────────────────────────────────
!sudo apt-get update && sudo apt-get install zstd -y
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (3,258 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (

In [101]:
# ── Start Ollama service ───────────────────────────────────────────────────────
import subprocess
import time
import os

# CUDA_VISIBLE_DEVICES=0 ensures Ollama finds the GPU even on multi-GPU hosts.
# Ollama auto-detects CUDA; no extra flags needed if the runtime has a GPU.
env = os.environ.copy()
env['CUDA_VISIBLE_DEVICES'] = '0'

ollama_proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env=env,
)
time.sleep(5)  # give it time to start
print('Ollama running')

Ollama running


In [102]:
import subprocess

# Get list of already pulled models to avoid redundant downloads
existing_models_raw = subprocess.check_output(['ollama', 'list'], text=True)
existing_models = [line.split()[0] for line in existing_models_raw.strip().split('\n')[1:]]

# Combine models and grader models for pulling
target_models = set(MODELS + GRADER_MODEL)

for model in target_models:
    base = model.removeprefix('openai/')
    if base in existing_models:
        print(f'Model {base} already exists, skipping pull.')
    else:
        print(f'Pulling {base}...')
        !ollama pull {base}

# Verify GPU offloading status
!ollama ps

Pulling mistral:7b...

Model gemma3:270m already exists, skipping pull.
NAME    ID    SIZE    PROCESSOR    CONTEXT    UNTIL 


In [103]:
# ── Install Python dependencies ────────────────────────────────────────────────
!pip install -q inspect-ai openai deep-translator python-dotenv

In [104]:
# ── Point to local Ollama instead of OpenWebUI ────────────────────────────────
import os
os.environ['OPENAI_API_KEY'] = 'ollama'
os.environ['OPENAI_BASE_URL'] = 'http://localhost:11434/v1'

In [ ]:
# ── Run evaluation ─────────────────────────────────────────────────────────────
import subprocess
import os

result = subprocess.run(
    ['python', 'eval.py'],
    cwd=f'{PROJECT_PATH}',
    env=os.environ.copy(),
    capture_output=True,
    text=True
)

print('Exit code:', result.returncode)
if result.stdout:
    print('\n--- Standard Output ---')
    print(result.stdout)
if result.stderr:
    print('\n--- Standard Error ---')
    print(result.stderr)